# Program 4: Train a custom Word2Vec model on a domain corpus

**Aim.** Train embeddings on a small medical corpus and analyse how they capture domain-specific meaning.

**Theory.** Word2Vec (skip-gram) trains a shallow network to predict context words from a centre word; the learned weights are the embeddings. A small corpus gives noisy vectors, so results are compared with GloVe, which was trained on billions of words.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import re, random
from pathlib import Path
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from src.tagger.synth import make_lines, split_names

# corpus 1: the project's 10 synthetic patient leaflets
leaflets = ' '.join(p.read_text(encoding='utf-8') for p in sorted((ROOT / 'data' / 'leaflets').glob('*.md')))
sent1 = [simple_preprocess(s) for s in re.split(r'[.\n]', leaflets) if len(s.split()) > 3]
# corpus 2: 4,000 synthetic dosing sentences built from invented prescription lines
names, _ = split_names()
sent2 = []
for ln in make_lines(4000, names, seed=3):
    g = ln.gold
    sent2.append(simple_preprocess(f"take {g.get('name','')} {g.get('dose','')} {g.get('frequency','')} {g.get('instructions','')} for {g.get('duration','')}"))
corpus = sent1 + sent2
print(len(sent1), 'leaflet sentences +', len(sent2), 'dosing sentences =', len(corpus), '; tokens:', sum(map(len, corpus)))

277 leaflet sentences + 4000 dosing sentences = 4277 ; tokens: 30150


In [3]:
model = Word2Vec(corpus, vector_size=64, window=5, min_count=3, sg=1, epochs=40, seed=1, workers=1)
print('vocabulary:', len(model.wv))
for w in ['fever', 'food', 'tablet', 'paracetamol', 'daily']:
    if w in model.wv:
        print(f'{w:12s}', ', '.join(f'{x} ({s:.2f})' for x, s in model.wv.most_similar(w, topn=5)))

vocabulary: 493
fever        pain (0.70), breathless (0.56), cough (0.55), dinner (0.53), thinly (0.50)
food         hour (0.68), breakfast (0.64), milk (0.61), half (0.59), meals (0.59)
tablet       glass (0.90), without (0.80), time (0.79), same (0.77), bisacodyl (0.68)
paracetamol  these (0.86), reactions (0.84), flu (0.83), rarely (0.83), relieves (0.83)
daily        day (0.69), bd (0.69), tds (0.69), hs (0.66), od (0.64)


In [4]:
# does the model place medicine names together? share of each drug's 5 neighbours that are also drug names
from src.tagger.synth import ALL_NAMES
drugs = {n.lower() for n in ALL_NAMES}
rows = []
for d in ['paracetamol', 'amoxicillin', 'metformin', 'cetirizine', 'atorvastatin', 'pantoprazole']:
    if d in model.wv:
        nb = [w for w, _ in model.wv.most_similar(d, topn=5)]
        rows.append((d, sum(w in drugs for w in nb) / 5))
print('share of neighbours that are medicines:', {d: v for d, v in rows}, ' mean', round(sum(v for _, v in rows) / len(rows), 2))
import gensim.downloader as api
glove = api.load('glove-wiki-gigaword-100')
rows = []
for d in ['paracetamol', 'amoxicillin', 'metformin', 'cetirizine', 'atorvastatin', 'pantoprazole']:
    if d in glove:
        rows.append(sum(w in drugs for w, _ in glove.most_similar(d, topn=5)) / 5)
print('GloVe, same measure, mean', round(sum(rows) / len(rows), 2), 'over', len(rows), 'drugs it knows')

share of neighbours that are medicines: {'paracetamol': 0.0, 'metformin': 0.8, 'cetirizine': 1.0, 'atorvastatin': 1.0, 'pantoprazole': 1.0}  mean 0.76


GloVe, same measure, mean 0.4 over 6 drugs it knows


## Observations

Even a tiny domain corpus teaches the model that medicine names behave alike (they appear in the same slots of a dosing sentence). That is partly because the dosing sentences are built from a template, so the result flatters the model; with only a few thousand sentences the vectors for rare words are unreliable. GloVe knows more words but nothing about this project's usage.